# Finalize SAM 3 annotations and prepare the segmentation dataset

Run this notebook after `05_sam3_batch_annotation_from_processed.ipynb` has processed the entire saved selection and a person has reviewed every overlay.

This notebook:

1. verifies that batch annotation is complete;
2. records human review decisions;
3. audits every accepted image/mask pair;
4. writes annotation summary reports;
5. converts accepted binary masks to YOLO Segmentation polygons;
6. preserves the video-level train/validation/test split from the original manifest.

It does not train a model yet.

## 1. Mount Drive and load annotation records

In [ ]:
from pathlib import Path
from datetime import datetime
import json, shutil
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/CMZprojectphase2')
ANNOTATION_ROOT = PROJECT_ROOT / 'data' / 'sam3_batch_annotation'
SELECTION_CSV = ANNOTATION_ROOT / 'annotation_selection.csv'
REVIEW_CSV = ANNOTATION_ROOT / 'review_status.csv'
PROPOSAL_DIR = ANNOTATION_ROOT / 'proposals'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'civet_yolo_segmentation'

for required in [SELECTION_CSV, REVIEW_CSV, PROPOSAL_DIR]:
    if not required.exists():
        raise FileNotFoundError(required)

selection = pd.read_csv(SELECTION_CSV)
review = pd.read_csv(REVIEW_CSV)
print('Project root:', PROJECT_ROOT)
print('Selected:', len(selection))
print('Completed proposals:', len(review))
print('Dataset output:', DATASET_ROOT)

## 2. Require complete batch processing

Do not finalize a partial first batch. If this cell fails, return to section 5 of notebook `05` and run the pending/batch cells repeatedly.

In [ ]:
selected_ids = set(selection['selection_id'].astype(str))
completed_ids = set(review['selection_id'].astype(str))
missing_ids = sorted(selected_ids - completed_ids)
extra_ids = sorted(completed_ids - selected_ids)

print('Missing proposals:', len(missing_ids))
print('Unexpected proposals:', len(extra_ids))
if missing_ids:
    print('First missing IDs:', missing_ids[:20])
    raise RuntimeError('Batch annotation is incomplete. Finish notebook 05 before continuing.')
if extra_ids:
    raise RuntimeError(f'Review CSV contains IDs outside the saved selection: {extra_ids[:20]}')
print('Batch annotation is complete.')

## 3. Record the human review

The user confirmed that all proposed masks were visually reviewed and correct, so `mask_proposed` rows are changed to `accepted`.

Rows where SAM returned no mask are **not** automatically accepted as negatives. Set `CONFIRM_NO_DETECTIONS_ARE_NEGATIVE = True` only after opening every such source image and confirming that no civet is visible.

In [ ]:
CONFIRM_ALL_PROPOSED_MASKS_REVIEWED = True
CONFIRM_NO_DETECTIONS_ARE_NEGATIVE = False
REVIEWER = 'project_team'
reviewed_at = datetime.now().isoformat(timespec='seconds')

if not CONFIRM_ALL_PROPOSED_MASKS_REVIEWED:
    raise RuntimeError('Set CONFIRM_ALL_PROPOSED_MASKS_REVIEWED=True only after checking every overlay.')

mask_found = review['proposal_result'].eq('mask_proposed')
review.loc[mask_found, 'mask_status'] = 'accepted'
review.loc[mask_found, 'review_note'] = 'Pixel mask visually reviewed and accepted.'
review.loc[mask_found, 'reviewer'] = REVIEWER
review.loc[mask_found, 'reviewed_at'] = reviewed_at

no_detection = review['proposal_result'].eq('no_sam_detection')
if CONFIRM_NO_DETECTIONS_ARE_NEGATIVE:
    review.loc[no_detection, 'mask_status'] = 'negative'
    review.loc[no_detection, 'review_note'] = 'Human reviewed the source image; no civet is visible.'
    review.loc[no_detection, 'reviewer'] = REVIEWER
    review.loc[no_detection, 'reviewed_at'] = reviewed_at
else:
    review.loc[no_detection, 'mask_status'] = 'needs_review'
    review.loc[no_detection, 'review_note'] = 'No SAM mask; source image still requires human negative confirmation.'

review.to_csv(REVIEW_CSV, index=False)
print('Saved final review decisions:', REVIEW_CSV)
display(review['mask_status'].value_counts(dropna=False).rename('images').to_frame())

## 4. Audit accepted masks

An accepted mask must exist, match the image dimensions, contain only 0/255, and contain foreground pixels. Any problem blocks dataset export.

In [ ]:
problems = []
audit_rows = []

for row in review.itertuples():
    if row.mask_status not in {'accepted', 'corrected'}:
        continue
    item_dir = PROPOSAL_DIR / str(row.selection_id)
    image_path = item_dir / 'image.jpg'
    mask_path = item_dir / 'mask.png'
    metadata_path = item_dir / 'metadata.json'

    missing = [p.name for p in [image_path, mask_path, metadata_path] if not p.exists()]
    if missing:
        problems.append({'selection_id': row.selection_id, 'problem': f'missing files: {missing}'})
        continue

    image = Image.open(image_path).convert('RGB')
    mask_array = np.asarray(Image.open(mask_path).convert('L'))
    values = set(np.unique(mask_array).tolist())
    item_problems = []
    if image.size != (mask_array.shape[1], mask_array.shape[0]):
        item_problems.append(f'size mismatch image={image.size}, mask={mask_array.shape[::-1]}')
    if not values.issubset({0, 255}):
        item_problems.append(f'invalid mask values: {sorted(values)[:20]}')
    foreground = int((mask_array > 0).sum())
    if foreground == 0:
        item_problems.append('empty foreground')
    if item_problems:
        for message in item_problems:
            problems.append({'selection_id': row.selection_id, 'problem': message})
    audit_rows.append({
        'selection_id': row.selection_id, 'width': image.width, 'height': image.height,
        'foreground_pixels': foreground, 'foreground_fraction': foreground / mask_array.size,
        'mask_values': ','.join(map(str, sorted(values))),
    })

audit = pd.DataFrame(audit_rows)
problems_df = pd.DataFrame(problems, columns=['selection_id', 'problem'])
audit.to_csv(ANNOTATION_ROOT / 'accepted_mask_audit.csv', index=False)
problems_df.to_csv(ANNOTATION_ROOT / 'accepted_mask_problems.csv', index=False)
print('Accepted/corrected masks audited:', len(audit))
print('Problems:', len(problems_df))
if len(problems_df):
    display(problems_df.head(30))
    raise RuntimeError('Fix accepted-mask problems before dataset export.')
print('All accepted masks passed structural validation.')

## 5. Generate annotation reports

In [ ]:
review_with_selection = review.merge(
    selection.drop(columns=[c for c in ['video_id', 'label', 'lighting', 'timestamp_sec', 'annotation_kind'] if c in selection.columns]),
    on='selection_id', how='left', validate='one_to_one',
)

summary = (review.groupby(['behavior', 'lighting', 'mask_status'], dropna=False)
           .size().rename('images').reset_index())
per_video = (review.groupby(['video_id', 'behavior', 'lighting', 'mask_status'], dropna=False)
             .size().rename('images').reset_index())
summary.to_csv(ANNOTATION_ROOT / 'annotation_review_summary.csv', index=False)
per_video.to_csv(ANNOTATION_ROOT / 'annotation_per_video.csv', index=False)
display(summary)

accepted = review['mask_status'].isin(['accepted', 'corrected'])
mean_score = pd.to_numeric(review.loc[accepted, 'score'], errors='coerce').mean()
counts = review['mask_status'].value_counts()
report = f'''# CMZ SAM 3 Pixel Annotation Report

## Dataset and review

- Selected frames: {len(selection)}
- Completed SAM proposals: {len(review)}
- Accepted/corrected masks: {int(accepted.sum())}
- Human-confirmed negatives: {int(counts.get('negative', 0))}
- Needs review: {int(counts.get('needs_review', 0))}
- Rejected: {int(counts.get('rejected', 0))}
- Mean SAM confidence for accepted/corrected masks: {mean_score:.4f}

## Method

Frames were selected from the existing processed CMZ dataset using `metadata/manifest.csv` and `processed/full640`. These images use the project's grayscale three-channel, denoising, CLAHE and letterbox preprocessing. SAM 3 used the text prompt `animal` at confidence threshold 0.5. Every accepted mask was visually reviewed.

## Label definition

Binary PNG masks use 255 for visible civet pixels and 0 for background. Behavior labels (Eating, Locomotion and Resting) come from the source manifest and remain subject to human label review.

## Limitations

SAM confidence is not boundary accuracy. Lighting and behavior are confounded in the small source dataset, some behavior-lighting combinations may be absent, and detector-miss images require separate negative review.

## Next step

Train a segmentation model on the exported video-level train/validation/test splits and evaluate mask IoU and Dice separately for daytime and infrared footage.
'''
REPORT_PATH = ANNOTATION_ROOT / 'annotation_report.md'
REPORT_PATH.write_text(report, encoding='utf-8')
print(report)
print('Saved report:', REPORT_PATH)

## 6. Prepare YOLO Segmentation directories

Accepted/corrected masks become class `0` (`civet`). Human-confirmed negative images receive an empty label file. Rows still marked `needs_review` or `rejected` are excluded.

The original manifest split is preserved. YOLO calls validation `val`; test remains test.

In [ ]:
if 'split' not in selection.columns:
    raise ValueError('annotation_selection.csv has no split column; regenerate selection from a manifest containing video-level splits.')

eligible = review.merge(
    selection[['selection_id', 'split', 'fold']],
    on='selection_id', how='left', validate='one_to_one',
) if 'fold' in selection.columns else review.merge(
    selection[['selection_id', 'split']], on='selection_id', how='left', validate='one_to_one'
)
eligible = eligible[eligible['mask_status'].isin(['accepted', 'corrected', 'negative'])].copy()
eligible['split'] = eligible['split'].replace({'validation': 'val'}).fillna('train')
unknown_splits = set(eligible['split']) - {'train', 'val', 'test'}
if unknown_splits:
    raise ValueError(f'Unsupported splits: {unknown_splits}')

if DATASET_ROOT.exists():
    raise FileExistsError(f'{DATASET_ROOT} already exists. Rename it or remove it deliberately before rebuilding.')
for split in ['train', 'val', 'test']:
    (DATASET_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)
print('Eligible images:', len(eligible))
display(pd.crosstab(eligible['mask_status'], eligible['split'], margins=True))

## 7. Convert binary masks to YOLO polygons

The largest external contour is used as the single civet instance. The export records how much foreground area that contour retains; values below 0.98 are flagged because they may indicate disconnected mask fragments.

In [ ]:
import cv2

def mask_to_yolo_polygon(mask_array, epsilon_fraction=0.001):
    binary = (mask_array > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        raise ValueError('No foreground contour')
    contour = max(contours, key=cv2.contourArea)
    perimeter = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon_fraction * perimeter, True).reshape(-1, 2)
    if len(approx) < 3:
        raise ValueError('Polygon has fewer than three points')
    h, w = binary.shape
    normalized = np.column_stack([approx[:, 0] / w, approx[:, 1] / h]).clip(0, 1)
    largest_area = float(cv2.contourArea(contour))
    foreground = float((binary > 0).sum())
    return normalized, largest_area / foreground if foreground else 0.0

export_rows, export_problems = [], []
for row in eligible.itertuples():
    item_dir = PROPOSAL_DIR / str(row.selection_id)
    source_image = item_dir / 'image.jpg'
    image_target = DATASET_ROOT / 'images' / row.split / f'{row.selection_id}.jpg'
    label_target = DATASET_ROOT / 'labels' / row.split / f'{row.selection_id}.txt'
    shutil.copy2(source_image, image_target)

    retained_fraction = np.nan
    polygon_points = 0
    if row.mask_status == 'negative':
        label_target.write_text('', encoding='utf-8')
    else:
        mask_array = np.asarray(Image.open(item_dir / 'mask.png').convert('L'))
        try:
            polygon, retained_fraction = mask_to_yolo_polygon(mask_array)
            polygon_points = len(polygon)
            coordinates = ' '.join(f'{value:.6f}' for point in polygon for value in point)
            label_target.write_text(f'0 {coordinates}\n', encoding='utf-8')
            if retained_fraction < 0.98:
                export_problems.append({'selection_id': row.selection_id, 'problem': f'largest contour retains {retained_fraction:.3f} of foreground'})
        except Exception as error:
            export_problems.append({'selection_id': row.selection_id, 'problem': str(error)})

    export_rows.append({
        'selection_id': row.selection_id, 'video_id': row.video_id,
        'behavior': row.behavior, 'lighting': row.lighting, 'split': row.split,
        'mask_status': row.mask_status, 'polygon_points': polygon_points,
        'largest_contour_retained_fraction': retained_fraction,
        'image_path': str(image_target.relative_to(DATASET_ROOT)),
        'label_path': str(label_target.relative_to(DATASET_ROOT)),
    })

export_manifest = pd.DataFrame(export_rows)
export_problem_df = pd.DataFrame(export_problems, columns=['selection_id', 'problem'])
export_manifest.to_csv(DATASET_ROOT / 'dataset_manifest.csv', index=False)
export_problem_df.to_csv(DATASET_ROOT / 'export_problems.csv', index=False)
print('Exported:', len(export_manifest))
print('Export warnings/problems:', len(export_problem_df))
if len(export_problem_df):
    display(export_problem_df.head(30))

## 8. Write dataset configuration and final checks

In [ ]:
yaml_text = f'''path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: civet
'''
(DATASET_ROOT / 'dataset.yaml').write_text(yaml_text, encoding='utf-8')

checks = []
for split in ['train', 'val', 'test']:
    images = sorted((DATASET_ROOT / 'images' / split).glob('*.jpg'))
    labels = sorted((DATASET_ROOT / 'labels' / split).glob('*.txt'))
    image_stems, label_stems = {p.stem for p in images}, {p.stem for p in labels}
    checks.append({'split': split, 'images': len(images), 'labels': len(labels), 'pairs_match': image_stems == label_stems})
checks = pd.DataFrame(checks)
display(checks)
if not checks['pairs_match'].all():
    raise RuntimeError('Image/label pairs do not match.')

dataset_report = (
    export_manifest.groupby(['split', 'lighting', 'behavior', 'mask_status'], dropna=False)
    .size().rename('images').reset_index()
)
dataset_report.to_csv(DATASET_ROOT / 'dataset_summary.csv', index=False)
display(dataset_report)
print('YOLO dataset:', DATASET_ROOT)
print('Config:', DATASET_ROOT / 'dataset.yaml')

## Completion criteria

The dataset is ready for a training notebook when:

- batch annotation was complete;
- accepted mask audit reported zero problems;
- every exported image has a matching label file;
- `needs_review` and `rejected` items were excluded;
- export warnings were inspected;
- train/validation/test contain different source videos according to the original manifest.

The next notebook should train a small segmentation model, report mask IoU and Dice/mAP separately for day and infrared footage, and retain the RGB/grayscale preprocessing decision in both training and inference.